# PneumoScan — model training and confidence gate

Trains and compares **seven models** on the Kaggle Chest X-Ray Images (Pneumonia) dataset,
then fits the confidence gate.

**Classical baselines:** logistic regression, RBF-SVM, random forest (HOG + downsampled pixels)
**Deep models:** small CNN from scratch, ResNet50, DenseNet121, EfficientNetB0 (two-stage fine-tuning)

### Before running
1. **Add Data** → search `chest-xray-pneumonia` (paultimothymooney)
2. **Settings → Accelerator → GPU T4 x2**
3. Run cells in order. Section 4 takes 40–70 min; everything else is minutes.

### What this pipeline does that most notebooks on this dataset do not
- splits by **patient**, not by image, so the same patient never appears in train and test
- discards the 16-image validation folder and re-splits train+val 80/20 by patient
- keeps a **classical baseline** so deep-model claims are relative to something
- fits both gate thresholds on **validation only**, then freezes them
- reports **bootstrap 95% confidence intervals**, not bare point estimates

## 1. Setup

In [34]:
import os, re, json, copy, time, hashlib, random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from sklearn.metrics import roc_auc_score, roc_curve

SEED = 42
IMG_SIZE = 224
BATCH = 32
EPOCHS_HEAD, EPOCHS_FT, PATIENCE = 4, 8, 3
TARGET_SENS = 0.90
RETAIN = 0.95   # share of validation images the gate keeps

DATA_ROOT = Path("/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray")
OUT = Path(os.environ.get("PNEUMOSCAN_OUT", "/kaggle/working/pneumoscan"))
OUT.mkdir(parents=True, exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed()
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV)
print("data root exists:", DATA_ROOT.exists())

device: cuda
data root exists: True


In [35]:
def confusion(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    return tp, tn, fp, fn

def metrics_at(y_true, probs, threshold=0.5):
    y_pred = (np.asarray(probs) >= threshold).astype(int)
    tp, tn, fp, fn = confusion(y_true, y_pred)
    sens = tp / (tp + fn) if tp + fn else float("nan")
    spec = tn / (tn + fp) if tn + fp else float("nan")
    prec = tp / (tp + fp) if tp + fp else float("nan")
    f1 = 2 * prec * sens / (prec + sens) if prec + sens else float("nan")
    try:
        auc = roc_auc_score(y_true, probs)
    except ValueError:
        auc = float("nan")
    return {"n": int(len(y_true)), "threshold": float(threshold),
            "sensitivity": sens, "specificity": spec, "precision": prec, "f1": f1,
            "accuracy": (tp + tn) / max(1, tp + tn + fp + fn), "auc": auc,
            "tp": tp, "tn": tn, "fp": fp, "fn": fn}

def bootstrap_ci(y_true, probs, threshold=0.5, metric="sensitivity", n=1000):
    rng = np.random.default_rng(SEED)
    y_true, probs = np.asarray(y_true), np.asarray(probs)
    idx, vals = np.arange(len(y_true)), []
    for _ in range(n):
        s = rng.choice(idx, size=len(idx), replace=True)
        if len(np.unique(y_true[s])) < 2:
            continue
        vals.append(metrics_at(y_true[s], probs[s], threshold)[metric])
    return tuple(float(v) for v in np.percentile(vals, [2.5, 97.5])) if vals else (np.nan, np.nan)

def expected_calibration_error(y_true, probs, bins=15):
    y_true, probs = np.asarray(y_true), np.asarray(probs)
    conf = np.where(probs >= 0.5, probs, 1 - probs)
    correct = ((probs >= 0.5).astype(int) == y_true).astype(float)
    edges, ece = np.linspace(0, 1, bins + 1), 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(ece)

def threshold_for_sensitivity(y_true, probs, target=TARGET_SENS):
    """Highest threshold that still reaches the target sensitivity."""
    best = 0.0
    for t in np.unique(np.round(np.asarray(probs), 4)):
        if metrics_at(y_true, probs, t)["sensitivity"] >= target:
            best = max(best, float(t))
    return best or 0.5

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def save_json(obj, name):
    with open(OUT / name, "w") as f:
        json.dump(obj, f, indent=2, default=float)
    print("wrote", OUT / name)

## 2. Data audit and patient-level split

Filenames carry the patient: `person53_bacteria_255.jpeg` → `person53`, and normal images
`IM-0115-0001.jpeg` → `IM-0115`. Splitting on images instead of patients is what inflates
the accuracy figures usually reported on this dataset.

In [36]:
CLASSES = {"NORMAL": 0, "PNEUMONIA": 1}

def patient_id(path):
    name = Path(path).name
    m = re.search(r"(person\d+)", name)
    if m: return m.group(1)
    m = re.search(r"(IM-\d+)", name)
    if m: return m.group(1)
    return Path(path).stem

def file_hash(path, blocksize=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while chunk := f.read(blocksize):
            h.update(chunk)
    return h.hexdigest()

rows = []
for folder in ("train", "val", "test"):
    d = DATA_ROOT / folder
    if not d.exists():
        print("missing:", d)
        continue
    for p in d.rglob("*"):
        if p.suffix.lower() not in {".jpeg", ".jpg", ".png"}:
            continue
        parts = {q.upper() for q in p.parts}
        if "NORMAL" in parts:
            label = 0
        elif "PNEUMONIA" in parts:
            label = 1
        else:
            continue
        rows.append((str(p), label, patient_id(p), folder))

df = pd.DataFrame(rows, columns=["path", "label", "patient_id", "orig_folder"])
print(f"found {len(df)} images, {df.patient_id.nunique()} patient groups")
print(df.groupby(["orig_folder", "label"]).size().unstack(fill_value=0))

found 5856 images, 2790 patient groups
label           0     1
orig_folder            
test          234   390
train        1341  3875
val             8     8


In [37]:
# duplicates
hashes = defaultdict(list)
for p in df["path"]:
    hashes[file_hash(p)].append(p)
dupe_paths = {p for ps in hashes.values() if len(ps) > 1 for p in ps[1:]}
print(f"duplicate files removed: {len(dupe_paths)}")
df = df[~df["path"].isin(dupe_paths)].reset_index(drop=True)

# image dimensions on a sample
sizes = [Image.open(p).size for p in df["path"].sample(min(400, len(df)), random_state=SEED)]
w, h = [s[0] for s in sizes], [s[1] for s in sizes]
print(f"width  {min(w)}–{max(w)} (median {int(np.median(w))})")
print(f"height {min(h)}–{max(h)} (median {int(np.median(h))})")

# patient-level re-split of train+val; original test folder untouched
pool = df[df.orig_folder.isin(["train", "val"])].copy()
test = df[df.orig_folder == "test"].copy(); test["split"] = "test"

pat_label = pool.groupby("patient_id")["label"].agg(lambda s: int(s.mean() >= 0.5))
shuffled = pd.Series(pat_label.index).sample(frac=1, random_state=SEED).tolist()
train_ids, val_ids = [], []
for cls in (0, 1):
    ids = [p for p in shuffled if pat_label[p] == cls]
    cut = int(0.8 * len(ids))
    train_ids += ids[:cut]; val_ids += ids[cut:]

train_set = set(train_ids)
pool["split"] = pool["patient_id"].map(lambda p: "train" if p in train_set else "val")
splits = pd.concat([pool, test], ignore_index=True)

leak = set(splits[splits.split == "train"].patient_id) & set(splits[splits.split == "val"].patient_id)
assert not leak, f"patient leakage: {leak}"
print("\nno patient appears in both train and val\n")

counts = splits.groupby(["split", "label"]).size().unstack(fill_value=0)
counts.columns = ["normal", "pneumonia"]
print(counts)
splits.to_csv(OUT / "splits.csv", index=False)

save_json({"images_used": int(len(splits)),
           "duplicates_removed": len(dupe_paths),
           "unique_patients": int(splits.patient_id.nunique()),
           "split_counts": counts.to_dict(),
           "width": {"min": min(w), "max": max(w)},
           "height": {"min": min(h), "max": max(h)}}, "audit_report.json")

duplicate files removed: 32
width  428–2518 (median 1264)
height 140–2364 (median 884)

no patient appears in both train and val

       normal  pneumonia
split                   
test      231        387
train    1095       3094
val       253        764
wrote /kaggle/working/pneumoscan/audit_report.json


In [38]:
# class distribution figure for the report
ax = counts.plot(kind="bar", figsize=(6, 3.4), color=["#7FA7D9", "#2F4B7C"], rot=0)
ax.set_ylabel("images"); ax.set_xlabel(""); ax.set_title("Class distribution after patient-level split")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig(OUT / "class_distribution.png", dpi=200); plt.show()

## 3. Classical baselines

Three models on HOG features plus downsampled pixels. These set the floor: every claim about
the deep models is stated relative to what a classical model already achieves on the same split.

In [39]:
from skimage.feature import hog
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

SMALL = 64

def features(path):
    with Image.open(path) as im:
        g = im.convert("L").resize((SMALL, SMALL))
    a = np.asarray(g, dtype=np.float32) / 255.0
    hg = hog(a, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2),
             feature_vector=True)
    return np.concatenate([hg, a[::2, ::2].ravel()])

def build_xy(split):
    sub = splits[splits.split == split]
    X = np.stack([features(p) for p in sub.path])
    return X, sub.label.to_numpy()

t0 = time.time()
Xtr, ytr = build_xy("train"); Xva, yva = build_xy("val"); Xte, yte = build_xy("test")
scaler = StandardScaler().fit(Xtr)
Xtr, Xva, Xte = scaler.transform(Xtr), scaler.transform(Xva), scaler.transform(Xte)
print(f"features ready in {(time.time()-t0)/60:.1f} min — {Xtr.shape[1]} dims")

features ready in 1.2 min — 2788 dims


In [40]:
classical_models = {
    "logistic_regression": LogisticRegression(max_iter=2000, class_weight="balanced",
                                              C=1.0, random_state=SEED),
    "svm_rbf": SVC(kernel="rbf", C=10.0, gamma="scale", probability=True,
                   class_weight="balanced", random_state=SEED),
    "random_forest": RandomForestClassifier(n_estimators=400, min_samples_leaf=2,
                                            class_weight="balanced_subsample",
                                            n_jobs=-1, random_state=SEED),
}

classical_results, classical_probs = {}, {}
for name, clf in classical_models.items():
    print(f"training {name} ...")
    clf.fit(Xtr, ytr)
    pva, pte = clf.predict_proba(Xva)[:, 1], clf.predict_proba(Xte)[:, 1]
    thr = threshold_for_sensitivity(yva, pva)          # validation only
    classical_results[name] = {"threshold": thr,
                               "val": metrics_at(yva, pva, thr),
                               "test": metrics_at(yte, pte, thr)}
    classical_probs[name] = pte
    m = classical_results[name]["test"]
    print(f"   test  sens={m['sensitivity']:.3f}  spec={m['specificity']:.3f}  auc={m['auc']:.3f}")

np.save(OUT / "y_test.npy", yte)
save_json(classical_results, "results_classical.json")

training logistic_regression ...
   test  sens=0.956  spec=0.680  auc=0.936
training svm_rbf ...
   test  sens=0.959  spec=0.736  auc=0.941
training random_forest ...
   test  sens=0.928  spec=0.835  auc=0.948
wrote /kaggle/working/pneumoscan/results_classical.json


## 4. Deep models

A small CNN from scratch as the deep baseline, then three ImageNet backbones fine-tuned in two
stages: head only at 1e-3, then the whole network at 1e-4. Early stopping on validation loss.

No horizontal flipping — it mirrors anatomy and produces images that are not valid radiographs.

⏱️ **This is the long cell: 40–70 minutes on a T4.**

In [41]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class CXR(Dataset):
    def __init__(self, frame, tf):
        self.paths, self.labels, self.tf = frame.path.tolist(), frame.label.tolist(), tf
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im:
            return self.tf(im.convert("RGB")), self.labels[i]

loaders = {
    "train": DataLoader(CXR(splits[splits.split == "train"], train_tf), batch_size=BATCH,
                        shuffle=True, num_workers=2, pin_memory=True),
    "val":   DataLoader(CXR(splits[splits.split == "val"], eval_tf), batch_size=BATCH, num_workers=2),
    "test":  DataLoader(CXR(splits[splits.split == "test"], eval_tf), batch_size=BATCH, num_workers=2),
}
ytr_deep = splits[splits.split == "train"].label.to_numpy()
pos_weight = torch.tensor([(ytr_deep == 0).sum() / max(1, (ytr_deep == 1).sum())],
                          dtype=torch.float32, device=DEV)
print("class imbalance handled with pos_weight =", float(pos_weight))

class imbalance handled with pos_weight = 0.35391080379486084


In [42]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        blocks, c_in = [], 3
        for c_out in (32, 64, 128, 128):
            blocks += [nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out),
                       nn.ReLU(inplace=True), nn.MaxPool2d(2)]
            c_in = c_out
        self.features = nn.Sequential(*blocks, nn.AdaptiveAvgPool2d(1))
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(128, 1))
    def forward(self, x):
        return self.classifier(self.features(x))
    def embed(self, x):
        return torch.flatten(self.features(x), 1)

def build(name):
    if name == "small_cnn":
        m = SmallCNN()
        return m, list(m.features.parameters()), list(m.classifier.parameters())
    if name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        head = nn.Linear(m.fc.in_features, 1); m.fc = head
    elif name == "densenet121":
        m = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        head = nn.Linear(m.classifier.in_features, 1); m.classifier = head
    elif name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        head = nn.Linear(m.classifier[1].in_features, 1); m.classifier[1] = head
    else:
        raise ValueError(name)
    head_ids = {id(p) for p in head.parameters()}
    backbone = [p for p in m.parameters() if id(p) not in head_ids]
    return m, backbone, list(head.parameters())

def embed(model, name, x):
    """Penultimate-layer features — the gate in section 5 is built on these."""
    if name == "small_cnn":
        return model.embed(x)
    if name == "resnet50":
        return torch.flatten(nn.Sequential(*list(model.children())[:-1])(x), 1)
    if name == "densenet121":
        f = torch.relu(model.features(x))
        return torch.flatten(F.adaptive_avg_pool2d(f, 1), 1)
    return torch.flatten(model.avgpool(model.features(x)), 1)

@torch.no_grad()
def infer(model, name, loader, with_features=False):
    model.eval()
    logits, labels, feats = [], [], []
    for x, y in loader:
        x = x.to(DEV, non_blocking=True)
        logits.append(model(x).squeeze(1).cpu().numpy())
        labels.append(np.asarray(y))
        if with_features:
            feats.append(embed(model, name, x).cpu().numpy())
    out = [np.concatenate(logits), np.concatenate(labels)]
    if with_features:
        out.append(np.concatenate(feats))
    return out

In [43]:
def run_epochs(model, name, params, lr, epochs, best_state, best_loss, patience):
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    cpu_crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight.cpu())
    for ep in range(epochs):
        model.train(); total = 0.0
        for x, y in loaders["train"]:
            x = x.to(DEV, non_blocking=True); y = y.float().unsqueeze(1).to(DEV)
            opt.zero_grad(); loss = crit(model(x), y); loss.backward(); opt.step()
            total += loss.item() * x.size(0)
        vlog, vlab = infer(model, name, loaders["val"])
        vloss = float(cpu_crit(torch.tensor(vlog).unsqueeze(1),
                               torch.tensor(vlab).float().unsqueeze(1)))
        print(f"      epoch {ep+1}: train {total/len(loaders['train'].dataset):.4f}  val {vloss:.4f}")
        if vloss < best_loss:
            best_loss, best_state, patience = vloss, copy.deepcopy(model.state_dict()), PATIENCE
        else:
            patience -= 1
            if patience <= 0:
                print("      early stopping"); break
    return best_state, best_loss, patience

DEEP = ["small_cnn", "resnet50", "densenet121", "efficientnet_b0"]
training_summary = {}

for name in DEEP:
    print(f"\n=== {name} ===")
    t0 = time.time()
    set_seed()
    model, backbone, head = build(name); model.to(DEV)
    best_state, best_loss, patience = None, float("inf"), PATIENCE

    if name == "small_cnn":
        best_state, best_loss, patience = run_epochs(
            model, name, list(model.parameters()), 1e-3,
            EPOCHS_HEAD + EPOCHS_FT, best_state, best_loss, patience)
    else:
        print("   stage 1: head only")
        for p in backbone: p.requires_grad = False
        best_state, best_loss, patience = run_epochs(
            model, name, head, 1e-3, EPOCHS_HEAD, best_state, best_loss, patience)
        print("   stage 2: fine-tuning")
        for p in backbone: p.requires_grad = True
        best_state, best_loss, patience = run_epochs(
            model, name, list(model.parameters()), 1e-4, EPOCHS_FT,
            best_state, best_loss, PATIENCE)

    model.load_state_dict(best_state)
    torch.save(best_state, OUT / f"{name}.pt")

    vlog, vlab, vfeat = infer(model, name, loaders["val"], True)
    tlog, tlab, tfeat = infer(model, name, loaders["test"], True)
    trlog, trlab, trfeat = infer(model, name, loaders["train"], True)
    np.savez(OUT / f"outputs_{name}.npz",
             val_logits=vlog, val_labels=vlab, val_feats=vfeat,
             test_logits=tlog, test_labels=tlab, test_feats=tfeat,
             train_feats=trfeat, train_labels=trlab)

    training_summary[name] = {
        "params_millions": round(sum(p.numel() for p in model.parameters()) / 1e6, 2),
        "best_val_loss": best_loss,
        "minutes": round((time.time() - t0) / 60, 1)}
    print(f"   {training_summary[name]['minutes']} min, "
          f"{training_summary[name]['params_millions']}M params")
    del model; torch.cuda.empty_cache()

save_json(training_summary, "training_summary.json")


=== small_cnn ===
      epoch 1: train 0.2139  val 0.1915
      epoch 2: train 0.1558  val 0.1502
      epoch 3: train 0.1386  val 0.1848
      epoch 4: train 0.1223  val 0.3378
      epoch 5: train 0.1139  val 0.1039
      epoch 6: train 0.0934  val 0.1449
      epoch 7: train 0.0859  val 0.0794
      epoch 8: train 0.0896  val 0.0807
      epoch 9: train 0.0935  val 0.2299
      epoch 10: train 0.0873  val 0.1173
      early stopping
   8.9 min, 0.24M params

=== resnet50 ===
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 223MB/s]


   stage 1: head only
      epoch 1: train 0.2331  val 0.1951
      epoch 2: train 0.1582  val 0.1575
      epoch 3: train 0.1369  val 0.1361
      epoch 4: train 0.1263  val 0.1239
   stage 2: fine-tuning
      epoch 1: train 0.0704  val 0.0454
      epoch 2: train 0.0397  val 0.0309
      epoch 3: train 0.0301  val 0.0334
      epoch 4: train 0.0232  val 0.0307
      epoch 5: train 0.0272  val 0.0285
      epoch 6: train 0.0149  val 0.0453
      epoch 7: train 0.0185  val 0.0290
      epoch 8: train 0.0119  val 0.0264
   12.0 min, 23.51M params

=== densenet121 ===
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 233MB/s]


   stage 1: head only
      epoch 1: train 0.2385  val 0.1790
      epoch 2: train 0.1460  val 0.1621
      epoch 3: train 0.1260  val 0.1163
      epoch 4: train 0.1212  val 0.1215
   stage 2: fine-tuning
      epoch 1: train 0.0587  val 0.0436
      epoch 2: train 0.0354  val 0.0761
      epoch 3: train 0.0257  val 0.0714
      epoch 4: train 0.0247  val 0.0304
      epoch 5: train 0.0228  val 0.0317
      epoch 6: train 0.0123  val 0.0340
      epoch 7: train 0.0120  val 0.0379
      early stopping
   11.0 min, 6.95M params

=== efficientnet_b0 ===
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 200MB/s]

   stage 1: head only


      epoch 1: train 0.2119  val 0.1471
      epoch 2: train 0.1391  val 0.1276
      epoch 3: train 0.1211  val 0.1179
      epoch 4: train 0.1133  val 0.1220
   stage 2: fine-tuning
      epoch 1: train 0.0704  val 0.0456
      epoch 2: train 0.0394  val 0.0547
      epoch 3: train 0.0362  val 0.0372
      epoch 4: train 0.0263  val 0.0362
      epoch 5: train 0.0254  val 0.0353
      epoch 6: train 0.0244  val 0.0420
      epoch 7: train 0.0183  val 0.0433
      epoch 8: train 0.0148  val 0.0238
   10.8 min, 4.01M params
wrote /kaggle/working/pneumoscan/training_summary.json


## 5. Calibration and the confidence gate

Three things, all fitted on **validation data only** and then frozen:

1. **Temperature scaling** — one parameter, so a stated confidence means what it claims (Guo et al., 2017)
2. **Mahalanobis distance** to class-conditional feature distributions — how unlike the training data an image is (Lee et al., 2018)
3. **Two thresholds** — confidence for the sensitivity target, distance for the retention rate

In [44]:
val_auc = {}
for name in DEEP:
    d = np.load(OUT / f"outputs_{name}.npz")
    val_auc[name] = metrics_at(d["val_labels"], sigmoid(d["val_logits"]))["auc"]
    print(f"{name:18s} val AUC {val_auc[name]:.4f}")

best = max(val_auc, key=val_auc.get)
print(f"\nselected model: {best}")

d = np.load(OUT / f"outputs_{best}.npz")
val_logits, val_labels, val_feats = d["val_logits"], d["val_labels"], d["val_feats"]
train_feats, train_labels = d["train_feats"], d["train_labels"]

small_cnn          val AUC 0.9857
resnet50           val AUC 0.9987
densenet121        val AUC 0.9986
efficientnet_b0    val AUC 0.9987

selected model: efficientnet_b0


In [45]:
def fit_temperature(logits, labels, steps=300):
    lg = torch.tensor(logits, dtype=torch.float32).unsqueeze(1)
    lb = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)
    log_t = torch.zeros(1, requires_grad=True)     # T = exp(log_t) stays positive
    crit, opt = nn.BCEWithLogitsLoss(), None
    opt = torch.optim.LBFGS([log_t], lr=0.05, max_iter=steps)
    def closure():
        opt.zero_grad(); loss = crit(lg / torch.exp(log_t), lb); loss.backward(); return loss
    opt.step(closure)
    return float(torch.exp(log_t).item())

T = fit_temperature(val_logits, val_labels)
raw, cal = sigmoid(val_logits), sigmoid(val_logits / T)
ece_before, ece_after = expected_calibration_error(val_labels, raw), expected_calibration_error(val_labels, cal)
print(f"temperature {T:.3f}   ECE {ece_before:.4f} -> {ece_after:.4f}")

def fit_mahalanobis(feats, labels):
    feats = np.asarray(feats, dtype=np.float64)
    means, centred = {}, []
    for c in (0, 1):
        f = feats[labels == c]; mu = f.mean(axis=0)
        means[c] = mu; centred.append(f - mu)
    cov = np.cov(np.concatenate(centred), rowvar=False)
    cov += np.eye(cov.shape[0]) * 1e-3
    return means, np.linalg.pinv(cov)

def mahalanobis(feats, means, prec):
    feats = np.asarray(feats, dtype=np.float64)
    ds = []
    for c in (0, 1):
        diff = feats - means[c]
        ds.append(np.einsum("ij,jk,ik->i", diff, prec, diff))
    return np.min(np.stack(ds, axis=1), axis=1)

means, prec = fit_mahalanobis(train_feats, train_labels)
val_dist = mahalanobis(val_feats, means, prec)
dist_threshold = float(np.quantile(val_dist, RETAIN))
conf_threshold = threshold_for_sensitivity(val_labels, cal, TARGET_SENS)

print(f"distance threshold   {dist_threshold:.1f}  (keeps {RETAIN:.0%} of validation)")
print(f"confidence threshold {conf_threshold:.4f} (for {TARGET_SENS:.0%} sensitivity)")

np.savez(OUT / "gate_stats.npz", class_means=np.stack([means[0], means[1]]), precision=prec)
gate = {"selected_model": best, "val_auc_by_model": val_auc, "temperature": T,
        "ece_before": ece_before, "ece_after": ece_after,
        "confidence_threshold": conf_threshold, "distance_threshold": dist_threshold,
        "retention_on_validation": RETAIN, "target_sensitivity": TARGET_SENS,
        "note": "fitted on validation only, then frozen"}
save_json(gate, "gate_params.json")

temperature 1.293   ECE 0.0077 -> 0.0058
distance threshold   2849.0  (keeps 95% of validation)
confidence threshold 0.9755 (for 90% sensitivity)
wrote /kaggle/working/pneumoscan/gate_params.json


In [46]:
# reliability diagram — before and after calibration
fig, axes = plt.subplots(1, 2, figsize=(9, 4), sharey=True)
for ax, p, title in zip(axes, [raw, cal], [f"before (ECE {ece_before:.3f})",
                                           f"after T={T:.2f} (ECE {ece_after:.3f})"]):
    edges = np.linspace(0, 1, 11)
    xs, ys = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (p > lo) & (p <= hi)
        if m.sum():
            xs.append(p[m].mean()); ys.append(val_labels[m].mean())
    ax.plot([0, 1], [0, 1], "k:", lw=0.8)
    ax.plot(xs, ys, "o-", color="#2F4B7C")
    ax.set_title(title); ax.set_xlabel("predicted probability")
axes[0].set_ylabel("observed frequency")
plt.tight_layout(); plt.savefig(OUT / "calibration.png", dpi=200); plt.show()

## 6. Results

The comparison table is the deliverable for your results chapter. Report what it shows —
if a classical model lands close to a deep one, that is a finding, not a failure.

In [47]:
rows, curves = [], {}
y_test = np.load(OUT / "y_test.npy")

for name in classical_probs:
    p, thr = classical_probs[name], classical_results[name]["threshold"]
    lo, hi = bootstrap_ci(y_test, p, thr)
    rows.append({"model": name, "family": "classical", **metrics_at(y_test, p, thr),
                 "sens_ci_low": lo, "sens_ci_high": hi})
    curves[name] = (y_test, p)

for name in DEEP:
    dd = np.load(OUT / f"outputs_{name}.npz")
    y = dd["test_labels"]
    p = sigmoid(dd["test_logits"] / (gate["temperature"] if name == best else 1.0))
    thr = gate["confidence_threshold"] if name == best else 0.5
    lo, hi = bootstrap_ci(y, p, thr)
    rows.append({"model": name, "family": "deep", **metrics_at(y, p, thr),
                 "sens_ci_low": lo, "sens_ci_high": hi})
    curves[name] = (y, p)

table = pd.DataFrame(rows)
table["params_millions"] = table.model.map(lambda n: training_summary.get(n, {}).get("params_millions"))
table = table.sort_values("auc", ascending=False).reset_index(drop=True)
table.to_csv(OUT / "model_comparison.csv", index=False)
table[["model", "family", "sensitivity", "specificity", "precision", "f1", "auc",
       "sens_ci_low", "sens_ci_high", "params_millions"]].round(4)

,model,family,sensitivity,specificity,precision,f1,auc,sens_ci_low,sens_ci_high,params_millions
0,densenet121,deep,0.9897,0.7749,0.8805,0.9319,0.9810,0.9783,0.9975,6.95
1,resnet50,deep,0.9948,0.6710,0.8351,0.9080,0.9733,0.9869,1.0000,23.51
2,efficientnet_b0,deep,0.9457,0.8701,0.9242,0.9349,0.9645,0.9221,0.9667,4.01
3,random_forest,classical,0.9276,0.8355,0.9043,0.9158,0.9482,0.8995,0.9510,NaN
4,svm_rbf,classical,0.9587,0.7359,0.8588,0.9060,0.9408,0.9371,0.9767,NaN
5,logistic_regression,classical,0.9561,0.6797,0.8333,0.8905,0.9358,0.9349,0.9750,NaN
6,small_cnn,deep,0.9819,0.5714,0.7933,0.8776,0.9229,0.9678,0.9945,0.24


In [48]:
plt.figure(figsize=(6, 5))
for name, (yy, p) in curves.items():
    fpr, tpr, _ = roc_curve(yy, p)
    plt.plot(fpr, tpr, lw=1.6, label=f"{name} ({roc_auc_score(yy, p):.3f})")
plt.plot([0, 1], [0, 1], "k:", lw=0.8)
plt.xlabel("1 − specificity"); plt.ylabel("sensitivity")
plt.title("ROC curves, held-out test set"); plt.legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT / "roc_curves.png", dpi=200); plt.show()

### Gated vs ungated

The headline number for the proposal: the **autonomy rate**, meaning the share of test images
resolved without escalation, and what sensitivity does on the cases the gate accepts.

In [49]:
d = np.load(OUT / f"outputs_{best}.npz")
y = d["test_labels"]
probs = sigmoid(d["test_logits"] / gate["temperature"])
dist = mahalanobis(d["test_feats"], means, prec)
conf = np.where(probs >= 0.5, probs, 1 - probs)

in_dist = dist <= gate["distance_threshold"]
confident = conf >= max(gate["confidence_threshold"], 1 - gate["confidence_threshold"])
accepted = in_dist & confident

ungated = metrics_at(y, probs, gate["confidence_threshold"])
gated = metrics_at(y[accepted], probs[accepted], gate["confidence_threshold"]) if accepted.sum() else {}

gating_results = {
    "selected_model": best,
    "ungated_all_images": ungated,
    "gated_accepted_only": gated,
    "accepted": int(accepted.sum()),
    "escalated_low_confidence": int((in_dist & ~confident).sum()),
    "rejected_out_of_distribution": int((~in_dist).sum()),
    "autonomy_rate": float(accepted.mean()),
}
save_json(gating_results, "gating_results.json")

print(f"autonomy rate            {accepted.mean():.1%}")
print(f"escalated (low conf.)    {(in_dist & ~confident).sum()}")
print(f"rejected (out of dist.)  {(~in_dist).sum()}")
print(f"\nsensitivity  all images {ungated['sensitivity']:.3f} -> accepted {gated.get('sensitivity', float('nan')):.3f}")
print(f"specificity  all images {ungated['specificity']:.3f} -> accepted {gated.get('specificity', float('nan')):.3f}")

wrote /kaggle/working/pneumoscan/gating_results.json
autonomy rate            73.9%
escalated (low conf.)    147
rejected (out of dist.)  14

sensitivity  all images 0.946 -> accepted 1.000
specificity  all images 0.870 -> accepted 0.706


In [50]:
cm = np.array([[ungated["tn"], ungated["fp"]], [ungated["fn"], ungated["tp"]]])
fig, ax = plt.subplots(figsize=(4.2, 3.8))
ax.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=14)
ax.set_xticks([0, 1], ["pred normal", "pred pneumonia"])
ax.set_yticks([0, 1], ["actual normal", "actual pneumonia"])
ax.set_title(f"{best} — test set")
plt.tight_layout(); plt.savefig(OUT / "confusion_best.png", dpi=200); plt.show()

## 7. Grad-CAM

Checks the model looks at lung fields rather than borders, text markers or scanner artefacts —
the shortcut learning Zech et al. (2018) documented.

In [51]:
if best == "small_cnn":
    print("Grad-CAM set up for the pretrained backbones; small_cnn selected, skipping.")
else:
    model, _, _ = build(best)
    model.load_state_dict(torch.load(OUT / f"{best}.pt", map_location=DEV))
    model.to(DEV).eval()
    target_layer = {"resnet50": lambda m: m.layer4[-1],
                    "densenet121": lambda m: m.features[-1],
                    "efficientnet_b0": lambda m: m.features[-1]}[best](model)

    store = {}
    target_layer.register_forward_hook(lambda m, i, o: store.__setitem__("act", o))
    target_layer.register_full_backward_hook(lambda m, gi, go: store.__setitem__("grad", go[0]))

    sample = splits[splits.split == "test"].groupby("label").head(3)
    fig, axes = plt.subplots(2, 3, figsize=(11, 7))
    for ax, (_, row) in zip(axes.ravel(), sample.iterrows()):
        with Image.open(row.path) as im:
            rgb = im.convert("RGB").resize((IMG_SIZE, IMG_SIZE))
        x = eval_tf(rgb).unsqueeze(0).to(DEV)
        out = model(x); model.zero_grad(); out.backward()
        wts = store["grad"].mean(dim=(2, 3), keepdim=True)
        cam = F.relu((wts * store["act"]).sum(1, keepdim=True))
        cam = F.interpolate(cam, (IMG_SIZE, IMG_SIZE), mode="bilinear",
                            align_corners=False)[0, 0].detach().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        p = float(torch.sigmoid(out / gate["temperature"]))
        ax.imshow(np.asarray(rgb)); ax.imshow(cam, cmap="jet", alpha=0.4)
        ax.set_title(f"{'pneumonia' if row.label else 'normal'} — p={p:.2f}", fontsize=10)
        ax.axis("off")
    plt.tight_layout(); plt.savefig(OUT / "gradcam_samples.png", dpi=200); plt.show()

/tmp/ipykernel_49/4109852662.py:27: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  p = float(torch.sigmoid(out / gate["temperature"]))


In [52]:
import shutil
shutil.make_archive("/kaggle/working/pneumoscan_outputs", "zip",
                    "/kaggle/working/pneumoscan")
print("ready")

ready


## 8. Summary and next steps

**Outputs** are saved under `/kaggle/working/pneumoscan/`: `model_comparison.csv`,
`gating_results.json`, `gate_params.json`, `audit_report.json`, `splits.csv`, and
the figures (`roc_curves.png`, `calibration.png`, `confusion_best.png`,
`gradcam_samples.png`, `class_distribution.png`).

**Findings so far.** On the held-out test set the pretrained models outperform the
classical baselines and the scratch CNN on AUC. The confidence gate resolves 73.9% of
images autonomously, raising sensitivity on accepted cases from 0.946 to 1.000 at the
cost of specificity (0.870 to 0.706).

**Not yet done.** The gate has been fitted and tested on in-distribution data only.
Evaluating it properly requires passing three kinds of out-of-distribution input
through the same frozen thresholds and reporting the share correctly rejected:

1. adult chest X-rays (RSNA Pneumonia Detection Challenge)
2. degraded versions of the test images (blur, exposure, rotation, cropping)
3. non-chest images, representing upload errors

This is the next stage of the project.